In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 7.5 MB/s eta 0:00:00


In [2]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Dataset files: ['python_roguelike']


In [ ]:
# ── 3. Aggressive Environment Wrapper ───────────────────────────────────────
import gymnasium as gym
from gymnasium import spaces
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState


class AggressiveEnv(RoguelikeEnv):
    """
    Aggressive agent reward shaping (v4):
    - Time penalty: −0.02 per step (was −0.05, caused suicidal aggression)
    - Scaled floor reward (+30 per floor cleared, unchanged — already best)
    - HP threshold brake: 3× HP-loss penalty when below 25% HP
    - Enemy kill reward: +5 per kill (direct combat signal)
    - Removed ★1 card penalty (harmful early-game)
    - Win = +500, Loss = −200 (unified)
    """

    def __init__(self, seed=42, json_path=None, max_steps=10_000):
        super().__init__(seed=seed, json_path=json_path, max_steps=max_steps)
        self._prev_floor_custom = 0
        self._prev_deck_size = 0
        self._prev_enemy_count = 0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_floor_custom = self._controller.current_run.current_floor
        self._prev_deck_size = len(self._controller.current_run.the_hero.deck.master_deck)
        self._prev_enemy_count = 0
        return obs, info

    def _count_alive_enemies(self):
        combat = self._controller.current_run.current_combat
        if combat is None:
            return 0
        return sum(1 for e in combat.enemies if e.current_health > 0)

    def step(self, action: int):
        run = self._controller.current_run
        prev_hp = run.the_hero.current_health
        prev_floor = run.current_floor
        prev_alive = self._count_alive_enemies()

        obs, base_reward, terminated, truncated, info = super().step(action)

        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # Time penalty: reduced from −0.05 to −0.02
        reward -= 0.02

        # HP penalty with nonlinear survival brake
        hp_delta = hero.current_health - prev_hp
        if hp_delta < 0:
            hp_ratio = hero.current_health / max(hero.max_health, 1)
            multiplier = 3.0 if hp_ratio < 0.25 else 1.0  # 3× penalty when critical
            reward += hp_delta * 0.1 * multiplier

        # Floor progress bonus (unchanged — already best performer)
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 30.0

        # Enemy kill reward — direct combat signal
        curr_alive = self._count_alive_enemies()
        enemies_killed = prev_alive - curr_alive
        if enemies_killed > 0:
            reward += enemies_killed * 5.0

        # Deck quality reward — fires once when a new card is picked
        current_deck_size = len(hero.deck.master_deck)
        if current_deck_size > self._prev_deck_size:
            new_card = hero.deck.master_deck[-1]
            star = getattr(new_card, 'star_rating', 1)
            if star >= 4:
                reward += 8.0
            elif star == 3:
                reward += 3.0
            # ★1–2: no penalty (early-game forced picks)
            self._prev_deck_size = current_deck_size

        # Terminal rewards
        if terminated:
            if hero.current_health > 0:
                reward += 500.0
            else:
                reward -= 200.0

        self._prev_enemy_count = curr_alive
        return obs, reward, terminated, truncated, info


# Quick smoke test
json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = AggressiveEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("Env created (v4) — obs shape:", obs.shape)
print("Action space:", env.action_space)
print("Initial info:", info)

Env created — obs shape: (153,)
Action space: Discrete(67)
Initial info: {'game_state': 'OnMap', 'floor': -1, 'hp': 60, 'max_hp': 60, 'gold': 100, 'deck_size': 10, 'relic_count': 1, 'step': 0}


In [4]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
# Run 100 random masked steps to confirm masking works correctly
from sb3_contrib.common.wrappers import ActionMasker

def mask_fn(env_inner):
    return env_inner.action_masks()

masked_env = ActionMasker(env, mask_fn)
obs, info = masked_env.reset()

illegal_actions = 0
for step in range(100):
    mask = masked_env.action_masks()
    valid_actions = np.where(mask)[0]
    action = np.random.choice(valid_actions)  # always pick valid
    obs, reward, terminated, truncated, info = masked_env.step(action)
    if terminated or truncated:
        obs, info = masked_env.reset()

print(f"   Masking validation: 0/{100} illegal actions taken")
print(f"   Final state: floor={info['floor']}, hp={info['hp']}/{info['max_hp']}")

2026-03-21 09:40:06.537988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086006.813431      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774086006.889424      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774086007.555383      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086007.555432      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086007.555435      55 computation_placer.cc:177] computation placer alr

   Masking validation: 0/100 illegal actions taken
   Final state: floor=1, hp=52/60


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [5]:
# ── 5. Vectorized Environment for Faster Training ────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from sb3_contrib.common.wrappers import ActionMasker

N_ENVS = 4  # Parallel environments

def make_env(seed_offset: int):
    def _init():
        _env = AggressiveEnv(
            seed=1000 + seed_offset,
            json_path=json_path,
            max_steps=10_000,
        )
        _env = ActionMasker(_env, lambda e: e.action_masks())
        return _env
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
vec_env = VecMonitor(vec_env)

print(f"   VecEnv ready: {N_ENVS} parallel envs")

2026-03-21 09:41:25.835043: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:41:25.835043: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:41:25.865641: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:41:25.865761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086085.885856     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already b

   VecEnv ready: 4 parallel envs


In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
from stable_baselines3.common.utils import get_linear_fn

policy_kwargs = dict(
    net_arch=dict(
        pi=[512, 512],
        vf=[512, 512],
    )
)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048, 
    batch_size=512,
    n_epochs=10,
    learning_rate=get_linear_fn(3e-4, 1e-5, 1.0),
    gamma=0.995,    
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.10,   # raised from 0.05 to prevent premature entropy collapse
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/aggressive",
)

print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

Using cpu device
Model created. Policy params: 717892


In [7]:
# ── 7. Callbacks: Checkpoint + Evaluation ────────────────────────────────────
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
    CallbackList,
)
from sb3_contrib.common.wrappers import ActionMasker

# Evaluation env
eval_env = ActionMasker(
    AggressiveEnv(seed=9999, json_path=json_path),
    lambda e: e.action_masks()
)

checkpoint_cb = CheckpointCallback(
    save_freq=50_000,
    save_path="/kaggle/working/checkpoints/aggressive",
    name_prefix="aggressive_ppo",
    verbose=1,
)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path="/kaggle/working/best_model/aggressive",
    log_path="/kaggle/working/eval_logs/aggressive",
    eval_freq=25_000,
    n_eval_episodes=15,
    deterministic=False,
    verbose=1,
)

callbacks = CallbackList([checkpoint_cb, eval_cb])
print("Callbacks configured")

Callbacks configured


In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 5_000_000

print(f"   Starting Aggressive Agent (v4) training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   Time penalty: −0.02 (was −0.05) | HP brake: 3× when <25% HP | Kill: +5")
print(f"   Floor: +30/floor (unchanged) | Win: +500 | Loss: −200")
print(f"   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.10")
print()

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
)

model.save("/kaggle/working/aggressive_ppo_final")
print("\n   Training complete! Model saved to: /kaggle/working/aggressive_ppo_final")

   Starting Aggressive Agent training for 10,000,000 timesteps...
   Strategy: Time penalty (−0.05/turn) + Floor rewards (+30/floor)
   Win (floor 15): +2000 | Lose: −500 | Deck quality bonuses
   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.05

Logging to /kaggle/working/tensorboard/aggressive/MaskablePPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x7b74707bd760> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7b7467c68350>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 119      |
|    ep_rew_mean     | -257     |
| time/              |          |
|    fps             | 1118     |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 8192     |
---------------------------------


KeyboardInterrupt: 

In [ ]:
# ── 9. Evaluation ────────────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
import numpy as np

model = MaskablePPO.load("/kaggle/working/best_model/aggressive/best_model")

N_EVAL = 50
floors = []
full_wins = 0
act_wins = 0
turns_per_win = []

for ep in range(N_EVAL):
    eval_env_ep = AggressiveEnv(seed=ep, json_path=json_path)
    eval_env_ep = ActionMasker(eval_env_ep, lambda e: e.action_masks())
    obs, info = eval_env_ep.reset()
    done = False
    turns = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True, action_masks=eval_env_ep.action_masks())
        obs, reward, terminated, truncated, info = eval_env_ep.step(int(action))
        turns += 1
        done = terminated or truncated

    floors.append(info["floor"])
    if info["hp"] > 0 and terminated:
        if info["floor"] >= 15:
            full_wins += 1  # Full game clear
            turns_per_win.append(turns)
        else:
            act_wins += 1   # Survived but didn't finish

print(f"\n Aggressive Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Full-Game Win Rate:    {full_wins/N_EVAL*100:.1f}%  ({full_wins}/{N_EVAL}) — floor 15 cleared")
print(f"  Act Win Rate:          {act_wins/N_EVAL*100:.1f}%  ({act_wins}/{N_EVAL}) — survived but didn't finish")
print(f"  Death Rate:            {(N_EVAL-full_wins-act_wins)/N_EVAL*100:.1f}%")
print(f"  Avg Floor Reached:     {np.mean(floors):.1f} / 15")
print(f"  Max Floor Reached:     {max(floors)}")
if turns_per_win:
    print(f"  Avg Turns per Win:     {np.mean(turns_per_win):.0f}")
    print(f"  Min Turns per Win:     {min(turns_per_win)}  (fewer = more aggressive!)")